[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/magnetotransport/magnetotransport.ipynb)

# Galvanomagnetic and Thermomagnetic Effects

This example computes the galvanomagnetic and thermomagnetic effects of a conductor holding two kinds of carrier, electrons and holes: how a magnetic field across it turns aside the charge and the heat they carry, along a Hall bar and around a Corbino ring, as the balance between the two kinds runs from electrons only, as in a doped semiconductor, through compensation, as in a semimetal such as bismuth, to holes only. Each carrier holds a charge and an entropy, so a fall of the electrochemical potential pushes it by its charge and a fall of temperature by its entropy, and it drifts where the drag of its collisions balances the push. The field turns each carrier's drift sideways as it moves, by its Hall angle, and the opposite charges of electrons and holes turn theirs opposite ways. With one kind of carrier, a current along the bar piles charge onto one side until the Hall voltage across the bar holds the carriers on course, and the bar's resistance along it stays as it was, while a fall of temperature alone moves nothing, the carriers held still by their own Seebeck voltage. At compensation the two kinds' turned currents cancel: no Hall voltage builds up to hold the carriers on course, and the resistance rises with the field, while both kinds are turned to the same side and carry their entropy there, which warms it, the Ettingshausen effect. A fall of temperature drives both kinds the same way, and the field parts them to opposite sides, where their charges build up a voltage across the bar, the Nernst effect, strongest exactly where the Hall voltage vanishes. A Corbino ring, its contacts along its rims, has no sides for anything to build up on: its carriers spiral, its resistance rises with the field whatever the balance, and a fall of temperature between its rims drives a current around it.

The algebra is planar spacetime joined with one degenerate direction for entropy, `Algebra("x+y+t-") * Algebra("s0")`, and a carrier is modelled as a parcel holding a charge along time and an entropy, `valence * (time + entropy_per_charge * entropy)`. The magnetic field is a plane of space, `strength * mv.xy`, and with the drive, minus the gradient of the potentials wedged with them, it pushes a moving carrier by one commutator, `(magnetic + drive).commutator(parcel + valence * velocity)`: through space the electric push, the push down the fall of temperature and the magnetic turning at once, with no cross product and no magnetic term of its own, and along time the power spent on the carrier, of which the magnetic field spends none. For a drift through space the field's commutator is their geometric product, so that a kind's steady drift is the push on it at rest turned by the inverse of one element, `1 - valence * mobility * magnetic`: a scalar plus a bivector, a rotor scaled by the secant of the kind's Hall angle, the angle flipping with the charge. A kind carries what it holds as one flux, `velocity.regressive(parcel)`, an antibivector holding its current of charge and its current of entropy together, and the conductor's response, the map from a drive to the flux it drives, sums the kinds' fluxes. One sum of turned pushes is then read two ways: through the planes that read charge, `mv.xt` and `mv.yt`, the opposite turns of electrons and holes cancel at compensation; through those that read entropy, `mv.xs` and `mv.ys`, which both kinds carry along their drift, they add. Reversing the field reverses each kind's rotor and turns the response into its adjugate, the Onsager–Casimir reciprocity as one identity between typed maps, and a drive's reading of the flux it drives, `drive & flux`, the dissipation, is never negative, a pairing every observer reads alike, so that a boost changes the electric field but not the entropy produced.

In the matrix notation of the two-band model, each band contributes the conductivity tensor $\sigma_i = \frac{n_i e\mu_i}{1 + \mu_i^2B^2}\begin{pmatrix}1 & \zeta_i\mu_iB\\ -\zeta_i\mu_iB & 1\end{pmatrix}$, with $\zeta_i = +1$ for holes and $-1$ for electrons, and the thermoelectric tensor $\alpha_i = S_i\sigma_i$ with its band's thermopower $S_i$; the conductor's resistivity is $\rho = (\sigma_e + \sigma_h)^{-1}$ and its thermopower tensor $S = \rho\,(\alpha_e + \alpha_h)$, whose off-diagonal part is the Nernst signal. At weak fields the Hall coefficient is $R_H = \frac{n_h\mu_h^2 - n_e\mu_e^2}{e\,(n_h\mu_h + n_e\mu_e)^2}$, which at compensation, $n_e = n_h$, with equal mobilities $\mu$ vanishes, while the resistivity held at one temperature rises as $\rho_{xx} = \rho_0(1 + \mu^2B^2)$, and a Corbino disc of radii $r_1 < r_2$ has the resistance $R = \frac{\ln(r_2/r_1)}{2\pi\sigma_{xx}}$. The Onsager–Casimir relations read $L_{ij}(\mathbf B) = L_{ji}(-\mathbf B)$, and Bridgman's relation $P\kappa = NT$ ties the Ettingshausen coefficient $P$ to the Nernst coefficient $N$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_animation
from examples.util.topology import Topology
from examples.thermodynamics.magnetotransport import render, scenarios

# Planar spacetime, and the entropy a parcel holds: a direction of its own, degenerate, since nothing measures an
# entropy against a length or a time.
ga = Algebra("x+y+t-") * Algebra("s0")
# The plane of space, and what a parcel holds, charge along time and entropy.
space = Algebra("x+y+")
amounts = Algebra("t-s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# A direction through space; the potentials, the electrochemical potential along t and the temperature along s; their
# drive; a flux; a carrier's velocity per drive; and the response, the map from a drive to the flux it drives.
Spatial = Plane.derive.within(space).derive.dual                              # Antivector-
Potential = Plane.derive.within(amounts)                                      # Vector+
Drive = ga.gatype.bivector().derive.touching(space, amounts)                  # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
Drifting = ga.gatype((Spatial, Drive))
Response = ga.gatype((Flux, Drive))
# A unit of charge, a direction along time, and a unit of entropy, the weight of a place.
time = -mv.t.dual()                                                           # [] Parcel-
entropy = mv.s.dual()                                                         # [] Parcel-

## The carriers' drift

Each kind of carrier drifts where the drag of its collisions, its velocity over its mobility, balances the whole push on it, Drude's steady state. That push is one commutator of the field and the drive with the carrier's moving parcel, `(magnetic + drive).commutator(parcel + valence * velocity)`, the Lorentz force and the push down the fall of temperature at once. Through space it is the drive's push on the parcel at rest, its charge down the fall of the electrochemical potential and its entropy down the fall of temperature, and the field's turning of the drift, which for a drift through space is their geometric product, by the parcel's charge; along time it is the power spent on the carrier, of which the magnetic field spends none. The balance, `velocity == mobility * (push + valence * magnetic * velocity)`, then makes the drift the push turned by the inverse of `hall`, one less the mobility times the field. A scalar plus a bivector of a plane of space is a rotor scaled by its size, so its inverse turns the push through the rotor's angle, the Hall angle, and shortens it by the angle's cosine; electrons and holes, opposite in charge, turn through opposite angles. The field's strength is measured in the inverse of the mobility, so that it is the tangent of the Hall angle.

In [ ]:
# The carriers, electrons and holes: the charge each holds; the entropy it carries per charge it holds, its
# thermopower, an electron's negative, so that both kinds carry entropy alike; and its mobility, the velocity a unit
# push gives it. The magnetic field's strength, the tangent of the Hall angle.
valences = np.array([-1, 1])
entropies_per_charge = np.array([-1.0, 1.0])
mobilities = np.array([1.0, 1.0])
strength = 2.0

In [ ]:
# Each kind's parcel, and the field, a plane of space.
parcels = valences * (time + entropies_per_charge * entropy)                 # [kinds] Parcel-
magnetic = strength * mv.xy                                                   # [] Bivector+
# A drive's push on each kind's parcel at rest, and one less the mobility times the field, by the parcel's charge: a
# rotor scaled by the secant of the kind's Hall angle, which turns the push into the drift.
push = Drive.commutator(parcels)                                              # [kinds] Spatial- <- Drive+
hall = 1 - valences * mobilities * magnetic                                   # [kinds] Even+
drifting = hall.inverse() * (mobilities * push)                               # [kinds] Spatial- <- Drive+
# Each kind's Hall angle: the logarithm of its rotor scaled to unit size is the angle times the field's plane.
hall_angles = hall.normalized().log().scalar_product(~mv.xy)                  # [kinds] Scalar+

print(f"Hall angles of the electrons and the holes: {np.degrees(hall_angles.to_array()).round(1)} degrees")

## Charge and entropy in one flux

A kind carries what it holds as one flux, its velocity joined with its parcel, `velocity.regressive(parcel)`, an antibivector in which its current of charge and its current of entropy sit side by side: the planes `mv.xt` and `mv.yt` read the charge it carries along the bar and across it, and `mv.xs` and `mv.ys` the entropy. A unit electric field along the bar drives a flux in each kind, the field turned by that kind's rotor; with half the carriers of each kind, at compensation, the flux is the mean of the two. Read for charge, the two turns cancel there; read for entropy, which the electrons carry against their charge, they add.

In [ ]:
# Each kind's flux per drive, at a unit density of it; and the flux a unit field along the bar drives with electrons only
# and with holes only.
carrying = drifting.regressive(parcels)                                       # [kinds] Flux- <- Drive+
field = mv.xt                                                                 # [] Drive+
end_fluxes = carrying(field)                                                  # [kinds] Flux-
render.draw_turns(field, end_fluxes);

The crimson arrow is the unit field along the bar, read through the planes each axis sweeps through time, `field & (axis & time)`. On the left the fluxes are read for charge: the electrons' current, grey, and the holes', black, are the field turned through opposite angles and shortened by their cosine, their tips on the circle with the field's arrow as its diameter. The dotted path joins them, and the current runs along it as the balance is swept; at compensation, halfway, the current runs along the field, the turns cancelled, shortened by the cosine squared. On the right the same fluxes are read for entropy: each kind's current carries its entropy per charge along, the Peltier effect, the holes' with their charge and the electrons' against it, so the electrons' arrow is reversed, and halfway the two add into a flux of entropy straight across the field, which the Ettingshausen effect piles up against a bar's side.

## The bar and the ring

The Hall bar is three widths long, with a contact across each end, and the Corbino ring's hole is four tenths of its outer radius, with a contact along each rim; both come from the scenario as nearly equilateral triangles, the bar's mirroring in its middle line, their vertices places of unit weight along the entropy. A corner's hat gradient is the plane through the facing side, along time and at unit entropy, over twice the triangle's area, so that it reads one at the corner. Pairs of probes on the bar, along its lower side a width apart about its middle and across its middle, read its voltages, and in the pictures the contacts are gold.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Plane]:
    """Each triangle's area `Scalar+[faces]`, and each corner's hat gradient `[corners] Plane+[faces]`."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, read through time at unit entropy.
    double_areas = ((sides[0] & sides[1]) & time) & entropy                   # Scalar-[faces]
    return double_areas.norm() / 2, (time & entropy & sides) / double_areas


# The bar, its contacts at its left end and its right and its probes' plus and minus terminals; the ring, its contacts
# along its inner rim and its outer.
bar_topology, bar_vertices, left, right, plus, minus = scenarios.bar()
ring_topology, ring_vertices, inner, outer = scenarios.ring()
bar_areas, bar_gradients = triangles(bar_topology, bar_vertices)             # Scalar+[faces], [corners] Plane+[faces]
ring_areas, ring_gradients = triangles(ring_topology, ring_vertices)         # Scalar+[faces], [corners] Plane+[faces]
render.draw_conductors(bar_topology, bar_vertices, (left, right), ring_topology, ring_vertices, (inner, outer),
                       scenarios.bar_resolution, scenarios.ring_resolution);

## Steady states under a current and under heat

A hat's gradient wedged with a flux, `gradients ^ flux`, is what the flux carries out of the corner's cell, charge and entropy at once. A rise of the potentials at a corner drives a flux through the response, which the other hats read as what it carries out of their cells; gathered onto the vertices, that is the map from the potentials to what each cell loses. In the steady state no free cell loses charge or entropy, the contacts held and the rest of the border insulated, so that charge and heat leave only through the contacts. The model is linear response: the entropy the drives produce, which grows as their square, is left out, so that the steady states are linear in the drives, and setting a current or opening the circuit takes only scaling and combining two solved drives, with no iteration. The conductor is driven two ways. Held at a fall of the electrochemical potential, both contacts at one temperature, it passes a current, and that drive is scaled to pass the current wanted. Held at a fall of temperature, it passes a thermoelectric current too, which the first drive, subtracted in proportion, brings to nothing, so that the conductor driven by heat passes no current, as with the circuit between its contacts open.

In [ ]:
def outflow(topology: Topology, areas: Scalar, gradients: Plane, response: Response) -> SparseExtensor:
    """What each vertex's cell loses as the potentials rise at each vertex, charge and entropy, `(Parcel- <-
    Potential+)[vertices <- vertices]`, under this response `[] Flux- <- Drive+`."""
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- vertices]
    # The drive of each hat's rise, and the other hats reading what the flux it drives carries out of their cells.
    driving = -(gradients ^ Potential)                                        # [corners] (Drive+ <- Potential+)[faces]
    local = areas * (gradients[:, None] ^ response(driving)[None, :])        # [corners, corners] (Parcel- <- Potential+)[faces]
    return (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)


def steady(topology: Topology, areas: Scalar, gradients: Plane, response: Response, high: np.ndarray, low: np.ndarray,
           current: float, temperature_difference: float) -> tuple[Potential, Potential]:
    """The conductor's potentials `Potential+[vertices]` under this response `[] Flux- <- Drive+`, its contacts at the
    vertices `high` and `low`: driven by this current, entering at `high`, both contacts at one temperature; and
    driven by heat, `high` warmer by this difference of temperature, no current passing."""
    shares = np.zeros(topology.n_vertices)
    shares[high], shares[low] = 1, -1
    towards_high = mv.scalar(shares[:, None]).field()                         # Scalar+[vertices]
    # A unit fall of the electrochemical potential and a unit fall of the temperature from `high` to `low`, held at the
    # contacts; and the reading of the free vertices.
    held = towards_high * stack((mv.t, mv.s)) / 2                             # [drives] Potential+[vertices]
    free = topology.reading(np.setdiff1d(np.arange(topology.n_vertices), np.concatenate([high, low])), 0)
    losing = outflow(topology, areas, gradients, response)                    # (Parcel- <- Potential+)[vertices <- vertices]
    loose = free * losing * ~free                                             # (Parcel- <- Potential+)[free <- free]
    potentials = held + ~free * loose.solve(-(free * losing(held)))           # [drives] Potential+[vertices]
    electric, thermal = potentials
    # The charge entering at the high contact, what its cells lose to the rest of the conductor.
    electric_current, thermal_current = mv.t & (topology.reading(high, 0) * losing(potentials)).sites.sum()
    return (electric * (current / electric_current),
            (thermal - electric * (thermal_current / electric_current)) * temperature_difference)

## Sweeping the carriers' balance

The balance of the carriers, the holes' share of them less the electrons', is swept from electrons only through compensation to holes only, in a field of fixed strength, with the carriers' total density held at one, so that the conductivity with no field is one at every balance. The lattice conducts entropy too, down a fall of temperature. At each balance the response sums each kind's flux per drive at its density, with the lattice's conduction, and the bar and the ring settle into their two steady states: driven by a unit current, and by a fall of temperature of one per unit length, which pushes a carrier's entropy as hard as the current's fall of the electrochemical potential pushes its charge. The Corbino ring's contacts are its whole border, so that nothing can build up across its carriers' way; its current runs from its inner rim to its outer, and its inner rim is the warmer. The next two cells take about four seconds, most of it drawing.

In [ ]:
# The lattice's conductivity, the entropy it conducts per unit fall of temperature; the current, and the fall of
# temperature per unit of the span between the contacts, the bar's length or the ring's width.
conductivity = 1.5
current, temperature_fall = 1.0, 1.0
# The balance, the holes' share of the carriers less the electrons', swept in a cosine so that it dwells at the ends,
# and each kind's density; the frames and their duration in milliseconds.
n_frames, frame_duration_ms = 75, 40
balances = -np.cos(np.linspace(0, np.pi, n_frames))                          # [frames]
densities = np.stack([(1 - balances) / 2, (1 + balances) / 2], axis=-1)     # [frames, kinds]

In [ ]:
# The response at each balance: each kind's flux per drive at its density, and the lattice conducting entropy down a
# fall of temperature.
conducting = conductivity * Drive.commutator(entropy).regressive(entropy)    # [] Flux- <- Drive+
responses = (densities * carrying).sum(axis=-1) + conducting                 # [frames] Flux- <- Drive+


def sweep(topology: Topology, areas: Scalar, gradients: Plane, high: np.ndarray, low: np.ndarray, span: float,
          densities: np.ndarray, responses: Response, driftings: Drifting
          ) -> Iterator[tuple[np.ndarray, Potential, Spatial, Potential, Spatial]]:
    """A conductor's steady states at each frame in turn, its contacts `high` and `low` this span apart, at the frame's
    densities `[frames, kinds]`, its response `[frames] Flux- <- Drive+` and each kind's velocity per drive in its field
    `[frames, kinds] Spatial- <- Drive+`: the densities `[kinds]`, and driven by the current and by heat, the
    potentials `Potential+[vertices]` and each kind's velocity `[kinds] Spatial-[faces]`."""
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- vertices]
    for frame_densities, response, frame_drifting in zip(densities, responses, driftings):
        electric, thermal = steady(topology, areas, gradients, response, high, low, current, temperature_fall * span)
        # Each triangle's drives: its corners' potentials on their hats' gradients.
        potentials = stack((electric, thermal))                               # [drives] Potential+[vertices]
        drives = -(gradients ^ (corners * potentials[:, None])).sum(axis=-1)  # [drives] Drive+[faces]
        yield frame_densities, electric, frame_drifting(drives[0]), thermal, frame_drifting(drives[1])


# The bar and the ring at each balance, the one field and each kind's velocity per drive in it the same in every frame;
# the ring's span between its contacts is its width.
balance_drifting = drifting.broadcast_to(densities.shape)                    # [frames, kinds] Spatial- <- Drive+
frames = list(zip(magnetic.broadcast_to(n_frames),
                  sweep(bar_topology, bar_areas, bar_gradients, left, right, scenarios.length, densities, responses,
                        balance_drifting),
                  sweep(ring_topology, ring_areas, ring_gradients, inner, outer, 1 - scenarios.inner_radius, densities,
                        responses, balance_drifting)))
bar_states = [bar_state for _, bar_state, _ in frames]
ring_states = [ring_state for _, _, ring_state in frames]
# Both driven by the current, the bar standing up beside the ring, on the scales of the first frame, electrons only.
conductors = (bar_topology, bar_vertices, (left, right), ring_topology, ring_vertices, (inner, outer),
              scenarios.bar_resolution, scenarios.ring_resolution)
show_animation(render.animate_current_driven(*conductors, frames[0], frames), "magnetotransport_charge",
               frame_duration_ms, n_frames=n_frames)

The bar stands on the left, turned so that its length runs up the picture: the current enters at the bottom, and the bar's lower side, along which the probes lie, is on the right. The ring is on the right, and both are driven by the current. Each is tinted by its temperature, grey in equilibrium, red where warmer and blue where cooler, with the level lines of its electrochemical potential and the two kinds' drifts streaked over it, the electrons' white and the holes' black, the more numerous kind inked in full and the other in proportion to its share; between them is the field's patch, its arc running the way the field turns. With electrons only, the bar's level lines lean across it by the Hall angle while the electrons run straight along it, held there by the Hall voltage. As holes join, the lean straightens, until at compensation the lines stand square across the bar and crowd together as its resistance rises, while both kinds, turned to the same side, carry their entropy into the lower side and warm it, the Ettingshausen effect. Past compensation the lines lean the other way, and with holes only they mirror the electrons'. In the ring, electrons spiral inwards and holes outwards, both turned the same way around, and nothing changes with the balance but which kind carries the current: the ring's level lines, and its resistance, one plus the strength squared times its resistance with no field, stay as they are at every balance. The next cell takes about four seconds, most of it drawing.

In [ ]:
# The same states driven by heat, on the same scales.
show_animation(render.animate_heat_driven(*conductors, frames[0], frames), "magnetotransport_heat", frame_duration_ms,
               n_frames=n_frames)

Driven by heat, with one kind of carrier, the bar's carriers stand still, held by the Seebeck voltage, whose level lines stand square across the bar; towards compensation the lines turn to run along it, the Nernst voltage building up across the bar while both kinds drift together down the fall of temperature, and past compensation they turn on, to stand across the bar again with the Seebeck voltage reversed. In the ring, one kind of carrier alone stands still in the Seebeck voltage between the rims; towards compensation that voltage fades, both kinds drift outwards, and the field turns them opposite ways around, so that a current circulates, the Nernst voltage that holds the bar's carriers having nowhere to build up.

## Voltages against the balance

A voltmeter across two probes reads the potential at its plus probe less that at its minus probe, a map from potentials to voltages. Driven by the current, the probes along the bar's lower side read its resistance along it, and those across its middle its Hall resistance, from the lower side to the upper. Driven by heat, the same probes read its Seebeck and its Nernst signal, the electric field along the bar and across it per fall of temperature along it: each voltage over the difference of temperature between the probes along the bar, with the sign of a field pointing down a fall. They also read its Righi–Leduc signal, the fall of temperature across the bar per fall along it, the heat the carriers carry turned across by the field. All five are read off the bar's states that the animations drew.

In [ ]:
electric = stack([potentials for _, potentials, _, _, _ in bar_states])      # [frames] Potential+[vertices]
thermal = stack([potentials for _, _, _, potentials, _ in bar_states])       # [frames] Potential+[vertices]
# A voltmeter across each pair of probes, along the bar's lower side and across its middle.
voltmeters = bar_topology.reading(plus, 0) - bar_topology.reading(minus, 0)        # Scalar+[pairs <- vertices]
electric_voltages = (voltmeters * (electric & time)).batch() / current       # [frames, pairs] Scalar+
thermal_voltages = (voltmeters * (thermal & time)).batch()                   # [frames, pairs] Scalar+
# Driven by heat, the differences of temperature between the probes, along the bar the upstream one the warmer.
temperatures = (voltmeters * (thermal & entropy)).batch()                    # [frames, pairs] Scalar+
warmer = temperatures[:, 0]                                                   # [frames] Scalar+
resistance, hall_resistance = electric_voltages[:, 0], electric_voltages[:, 1]   # [frames] Scalar+, [frames] Scalar+
seebeck, nernst = -thermal_voltages[:, 0] / warmer, -thermal_voltages[:, 1] / warmer   # [frames] Scalar+, [frames] Scalar+
righi_leduc = temperatures[:, 1] / warmer                                     # [frames] Scalar+
render.draw_readings(balances, resistance, hall_resistance, seebeck, nernst, righi_leduc);

At the top, driven by the current, the Hall resistance is minus the field's strength with electrons only and plus it with holes only, less the share the contacts short at the bar's middle, and passes through nothing at compensation; the resistance along the bar is one at either end, a square's resistance with no field, and peaks at compensation. The peak falls short of one plus the strength squared, which the bar would reach with its temperature held even: at compensation the difference of temperature the Ettingshausen effect builds up across the bar pushes both kinds back, doing part of the work that no Hall voltage can do there, since both carry entropy alike, while the lattice, conducting the entropy back, lets the rest of the turning through. In the middle, driven by heat, the Seebeck signal is each kind's own entropy per charge at either end, the entropy per charge its current carries as Peltier heat, so that Kelvin's relation, the Peltier coefficient the temperature times the Seebeck coefficient, holds by construction; the Seebeck signal passes through nothing at compensation, and the Nernst signal is nothing at either end, where the carriers stand still, and largest at compensation, exactly where the Hall resistance vanishes. At the bottom, the Righi–Leduc signal needs both kinds, unbalanced: with one kind the carriers stand still and only the lattice carries the heat, which the field does not turn, and at compensation both kinds drift straight down the fall of temperature, so it is nothing at the ends and in the middle, about a fifth between, and reversed with the balance.

## The field reversed

Reversing the magnetic field reverses each kind's rotor, `hall.reverse()`, and turns the response into its adjugate, the Onsager–Casimir reciprocity that the checks assert as one identity between typed maps; in a conductor the same in every direction, it makes each effect across the drive odd in the field, so that the effect changes sides with it. At compensation, driven by heat, the field is swept from the scenario's through nothing to the reversed. The next two cells take about five seconds, most of it drawing.

In [ ]:
# The field's strength swept from the scenario's through nothing to the reversed in a cosine, so that it dwells at
# either sense, and half the carriers of each kind in every frame.
reversal = strength * np.cos(np.linspace(0, np.pi, n_frames)) * mv.xy        # [frames] Bivector+
compensation = np.full_like(densities, 0.5)                                  # [frames, kinds]

In [ ]:
# Each kind's velocity per drive in each frame's field, and the response at compensation.
reversal_hall = 1 - valences * mobilities * reversal[:, None]                # [frames, kinds] Even+
reversal_drifting = reversal_hall.inverse() * (mobilities * push)            # [frames, kinds] Spatial- <- Drive+
reversal_responses = (compensation * reversal_drifting.regressive(parcels)).sum(axis=-1) + conducting   # [frames] Flux- <- Drive+
reversal_frames = zip(reversal,
                      sweep(bar_topology, bar_areas, bar_gradients, left, right, scenarios.length, compensation,
                            reversal_responses, reversal_drifting),
                      sweep(ring_topology, ring_areas, ring_gradients, inner, outer, 1 - scenarios.inner_radius,
                            compensation, reversal_responses, reversal_drifting))
# Both driven by heat, on the scales the balance's sweep is drawn on.
show_animation(render.animate_heat_driven(*conductors, frames[0], reversal_frames), "magnetotransport_reversed",
               frame_duration_ms, n_frames=n_frames)

The bar stands on the left, its warmer contact at the bottom, and the ring is on the right, both driven by heat at compensation, and the field's patch between them shrinks to a point as the field passes through nothing and grows back with its arc running the other way. In the ring the circulating current reverses with the field: the electrons, white, and the holes, black, drift outwards turned opposite ways around, straight outwards as the field passes through nothing, and then each turned the way the other was. In the bar the Nernst voltage reverses too, but its level lines, along the bar, are the same either way, since a level line does not say which side is the higher, and both kinds drift straight down the fall of temperature at every field, the Nernst voltage undoing the field's turning of them; only by the contacts, which short that voltage, are the two kinds parted to opposite sides, and there they swap sides.

## The effects by name

Each effect this example shows is named here with the scene or the reading where it appears.

| Effect | Where it appears |
| --- | --- |
| Hall | the current-driven bar's level lines, leaning by the Hall angle, and its Hall resistance |
| Corbino | the current-driven ring's spirals, and its resistance, one plus the strength squared at every balance |
| Nernst | the heat-driven bar's level lines along it at compensation, and its Nernst signal |
| Ettingshausen | the current-driven bar's lower side warmed at compensation, and the entropy read across the field in the turns |
| Righi–Leduc | the heat-driven bar's fall of temperature across it, between the ends and compensation |
| Seebeck | the heat-driven bar's level lines across it with one kind of carrier, and its Seebeck signal |
| Peltier | the entropy each kind's current carries, its entropy per charge, read in the turns |
| Kelvin | the Seebeck signal at either end, the entropy per charge the Peltier heat carries |
| Lorentz | the whole push on a moving carrier, `(magnetic + drive).commutator(parcel + valence * velocity)` |
| Drude | the drift where the drag balances that push, turned by the inverse of `1 - valence * mobility * magnetic` |
| Onsager | the dissipation `drive & flux`, never negative, and the response with no field its own adjugate, in the checks |
| Casimir | the ring's circulating current, reversed with the field, and the response in the reversed field, each rotor reversed, the adjugate of the response, in the checks |


In [ ]:
# checks
# Each kind's Hall angle is the angle whose tangent is the mobility times the field, the electrons' and the holes'
# opposite; at compensation the flux of a unit field carries its charge along the field and its entropy across it.
np.testing.assert_allclose(hall_angles.to_array(), -valences * np.arctan(mobilities * strength), atol=1e-12)
compensated = (end_fluxes[0] + end_fluxes[1]) / 2                             # [] Flux-
charge_along, charge_across = (mv.xt & compensated).to_array(), (mv.yt & compensated).to_array()
entropy_along, entropy_across = (mv.xs & compensated).to_array(), (mv.ys & compensated).to_array()
assert abs(charge_across) < 1e-12 * abs(charge_along) and abs(entropy_along) < 1e-12 * abs(entropy_across)
# Onsager-Casimir: reversing the field reverses each kind's rotor, and turns the response into its adjugate.
reversed_drifting = hall.reverse().inverse() * (mobilities * push)            # [kinds] Spatial- <- Drive+
reversed_responses = (densities * reversed_drifting.regressive(parcels)).sum(axis=-1) + conducting   # [frames] Flux- <- Drive+
scale = np.abs(responses.kernel).max()
np.testing.assert_allclose(reversed_responses.kernel, responses.adjugate().kernel, atol=1e-12 * scale)
# The dissipation is positive in every triangle of the bar at every balance under either drive, and the response's
# part odd in the field produces none of it.
bar_corners = bar_topology.incidence[2, 0]                                    # [corners] Scalar+[faces <- vertices]
bar_drives = -(bar_gradients ^ (bar_corners * stack((electric, thermal))[:, :, None])).sum(axis=-1)   # [drives, frames] Drive+[faces]
dissipation = bar_drives & responses(bar_drives)                             # [drives, frames] Scalar+[faces]
assert (dissipation > 0).all()
odd = (responses - reversed_responses) / 2                                    # [frames] Flux- <- Drive+
assert np.abs((bar_drives & odd(bar_drives)).to_array()).max() < 1e-12 * dissipation.to_array().max()
# The whole push on each drifting carrier in the current-driven bar at compensation is one commutator of the field and
# the drive with its moving parcel: read through space it is the drag, the velocity over the mobility; along time it is
# the power spent on the carrier, none of it the magnetic field's, and summed over the kinds at their densities it is
# the electric field's reading of the charge flux.
middle = n_frames // 2
_, compensated_electric, compensated_velocities, _, _ = bar_states[middle]
compensated_drive = bar_drives[0, middle]                                     # Drive+[faces]
compensated_flux = responses[middle](compensated_drive)                       # Flux-[faces]
moving_parcels = parcels + valences * compensated_velocities                  # [kinds] Point-[faces]
pushes = (magnetic + compensated_drive).commutator(moving_parcels)            # [kinds] Point-[faces]
through_space = stack((mv.x, mv.y))[:, None]                                  # [2, 1] Plane+
np.testing.assert_allclose((through_space & pushes).to_array(),
                           (through_space & (compensated_velocities / mobilities)).to_array(), atol=1e-10)
assert np.abs((mv.t & magnetic.commutator(moving_parcels)).to_array()).max() < 1e-12
electric_field = -(bar_gradients ^ ((bar_corners * (compensated_electric & time)) * mv.t)).sum(axis=0)   # Bivector+[faces]
power = (densities[middle] * (mv.t & pushes)).sum(axis=0)                     # Scalar+[faces]
np.testing.assert_allclose(power.to_array(), (electric_field & compensated_flux).to_array(), atol=1e-10)
# A boost along the bar mixes the magnetic field into the electric field across it that the moving observer reads, as
# Lorentz has it, and leaves the dissipation, the drive's reading of the flux, as it was.
rapidity = 0.3
boost = (mv.xt * (rapidity / 2)).exp()                                        # [] Rotor+
spacetime_field = electric_field + magnetic                                   # Bivector+[faces]
swept_across = mv.y.dual() & time                                             # [] Flux-
rest_reading = (spacetime_field & swept_across).to_array()
moving_reading = ((boost >> spacetime_field) & swept_across).to_array()
np.testing.assert_allclose(moving_reading, np.cosh(rapidity) * rest_reading + np.sinh(rapidity) * strength, atol=1e-10)
np.testing.assert_allclose(((boost >> compensated_drive) & (boost >> compensated_flux)).to_array(),
                           (compensated_drive & compensated_flux).to_array(), atol=1e-10)
# With one kind of carrier, the bar's resistance along it is a square's with no field, its Hall resistance the strength
# over the carriers' charge, less the contacts' share, its Seebeck signal the kind's entropy per charge and its Nernst
# signal nothing; at compensation the Hall resistance vanishes where the Nernst signal is largest.
np.testing.assert_allclose(resistance.to_array()[[0, -1]], 1.0, rtol=1e-2)
np.testing.assert_allclose(hall_resistance.to_array()[[0, -1]], strength / valences, rtol=2e-2)
np.testing.assert_allclose(seebeck.to_array()[[0, -1]], entropies_per_charge, rtol=1e-10)
np.testing.assert_allclose(nernst.to_array()[[0, -1]], 0.0, atol=1e-10)
assert abs(hall_resistance.to_array()[middle]) < 1e-12 * strength
assert np.abs(nernst.to_array()).argmax() == middle
# The Righi-Leduc signal needs both kinds, unbalanced: nothing at either end and at compensation, and of opposite
# signs on either side of compensation, the lower side the warmer where electrons are the more numerous.
np.testing.assert_allclose(righi_leduc.to_array()[[0, middle, -1]], 0.0, atol=1e-10)
assert righi_leduc.to_array()[middle // 2] > 0 > righi_leduc.to_array()[-1 - middle // 2]
# The ring's resistance is the same at every balance, one plus the strength squared times a ring's with no field, the
# logarithm of its radii's ratio over two pi.
ring_electric = stack([potentials for _, potentials, _, _, _ in ring_states])   # [frames] Potential+[vertices]
rim_voltmeter = ring_topology.reading(inner[:1], 0) - ring_topology.reading(outer[:1], 0)   # Scalar+[1 <- vertices]
ring_resistances = (rim_voltmeter * (ring_electric & time)).batch()[:, 0].to_array() / current   # [frames]
np.testing.assert_allclose(ring_resistances, ring_resistances[0], rtol=1e-10)
np.testing.assert_allclose(ring_resistances[0], (1 + strength ** 2) * np.log(1 / scenarios.inner_radius) / (2 * np.pi),
                           rtol=1e-2)
# Driven by heat, one kind of carrier alone stands still; at compensation no voltage builds up in the ring, and its two
# kinds circle its centre opposite ways: each centroid's offset from the centre joined with the drift there, read by
# the plane of the ring, summed over the triangles.
_, _, _, _, single_drifts = ring_states[0]
assert np.abs((stack((mv.x, mv.y)) & single_drifts[0].batch()[..., None]).to_array()).max() < 1e-10
_, _, _, compensated_potentials, compensated_drifts = ring_states[middle]
assert np.abs((compensated_potentials & time).to_array()).max() < 1e-12
ring_corners = ring_topology.incidence[2, 0]                                  # [corners] Scalar+[faces <- vertices]
offsets = ring_corners.mean(axis=0) * ring_vertices - entropy                 # Point-[faces]
circling = (ring_areas * (mv.xy & (offsets & compensated_drifts))).sites.sum().to_array()   # [kinds]
assert circling[0] * circling[1] < 0